In [51]:
import pandas as pd
import numpy as np

In [ ]:
df_m1 = pd.read_csv("m1_all.csv")
df_m1 = df_m1[df_m1["date"] >= '2021-01-01']
df_m2 = pd.read_csv("repo_module_daily_signals.csv")
df_m2 = df_m2[df_m2["date"] >= '2021-01-01']
df_m5 = pd.read_csv("m5_treasury_daily_signals.csv")
df_m5 = df_m5[df_m5["date"] >= '2021-01-01']
df_calendary = pd.read_csv("tax_calendar_all.csv", sep=';')
df_calendary = df_calendary[df_calendary["Дата"] >= '2021-01-01']
df_calendary = df_calendary.drop(columns={'Налоговые обязанности'})
priority = {'event': 0, 'holiday': 1, "plain": 1}
df_calendary['priority'] = df_calendary['Тип дня'].map(priority).fillna(99)
df_calendary = df_calendary.sort_values(['Дата', 'priority'])
df_calendary = df_calendary.drop_duplicates(subset='Дата', keep='first')
df_calendary = df_calendary.drop(columns='priority')

df_m4 = pd.DataFrame({"date": pd.date_range('2021-01-01', '2026-05-08')})
df_calendary["Дата"] = pd.to_datetime(df_calendary["Дата"])
event_dates = df_calendary.loc[df_calendary["Тип дня"] == "event", 'Дата']
all_marked_dates = pd.Series(dtype='datetime64[ns]')
for ed in event_dates:
    window_range = pd.date_range(ed - pd.Timedelta(days=1),
                                 ed + pd.Timedelta(days=1),
                                 freq='D')
    all_marked_dates = pd.concat([all_marked_dates, window_range.to_series()])

all_marked_dates = all_marked_dates.drop_duplicates()

df_m4['Tax_Week_Flag'] = df_m4["date"].isin(all_marked_dates).astype(int)
df_m4['End_of_Month_Flag'] = (
    (pd.to_datetime(df_m4['date']) + pd.offsets.MonthEnd(0))
    - pd.to_datetime(df_m4['date'])
).dt.days.lt(3).astype(int)
df_m4['End_of_Quarter_Flag'] = (
    (pd.to_datetime(df_m4['date']) + pd.offsets.QuarterEnd(0))  # дата конца квартала
    - pd.to_datetime(df_m4['date'])
).dt.days.lt(3).astype(int)

df_m4["date"] = pd.to_datetime(df_m4["date"])
df_m2["date"] = pd.to_datetime(df_m2["date"])
df_m1["date"] = pd.to_datetime(df_m1["date"])
df_m5["date"] = pd.to_datetime(df_m5["date"])
for val in ['mad_score_ruonia', 'mad_score_spread', 'MAD_score_Roskazna', 'MAD_score_CBR', 'MAD_score_rate_spread', "MAD_score_cover",'MAD_score_repo_volume']:
    if val in ['mad_score_ruonia', 'mad_score_spread']:
        mapping = df_m1.set_index('date')[val]
    if val in ["MAD_score_CBR","MAD_score_Roskazna"]:
        mapping = df_m5.set_index('date')[val]
    if val in ["MAD_score_cover","MAD_score_rate_spread","MAD_score_repo_volume"]:
        mapping = df_m2.set_index('date')[val]
    df_m4[val] = df_m4['date'].map(mapping)

mad_cols = [
    'mad_score_ruonia', 'MAD_score_Roskazna',
    'MAD_score_rate_spread', 'MAD_score_CBR', 'MAD_score_cover',
    'MAD_score_repo_volume'
]


df_m4['day_type'] = (
    df_m4['Tax_Week_Flag'].astype(str) +
    df_m4['End_of_Month_Flag'].astype(str) +
    df_m4['End_of_Quarter_Flag'].astype(str)
)

avg_by_type = df_m4.groupby('day_type')[mad_cols].mean()


tax_stress_by_type = avg_by_type.mean(axis=1)


max_stress = tax_stress_by_type.max()
alpha = 0.4 / max_stress if max_stress > 0 else 0.35 


seasonal_by_type = (1 + alpha * tax_stress_by_type).clip(1.0, 1.4)
seasonal_by_type.name = 'Seasonal_Factor'

df_m4['Seasonal_Factor'] = df_m4['day_type'].map(seasonal_by_type)
df_m4.drop(['mad_score_ruonia', 'mad_score_spread', 'MAD_score_Roskazna', 'MAD_score_CBR', 'MAD_score_rate_spread', "MAD_score_cover",'MAD_score_repo_volume', "day_type"], axis=1, inplace=True)

df_m4.to_csv("../../../ml-services/modules/m4_tax_saesonality/m1_all.csv")


C:\Users\Viktoria\AppData\Local\Temp\ipykernel_3480\1506597594.py:24: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  all_marked_dates = pd.concat([all_marked_dates, window_range.to_series()])


,date,Tax_Week_Flag,End_of_Month_Flag,End_of_Quarter_Flag,Seasonal_Factor
0,2021-01-01,0,0,0,1.400000
1,2021-01-02,0,0,0,1.400000
2,2021-01-03,0,0,0,1.400000
3,2021-01-04,0,0,0,1.400000
4,2021-01-05,0,0,0,1.400000
5,2021-01-06,0,0,0,1.400000
6,2021-01-07,0,0,0,1.400000
7,2021-01-08,0,0,0,1.400000
8,2021-01-09,0,0,0,1.400000
9,2021-01-10,1,0,0,1.348858
